# 作业三

## 2 卷积和池化层

### 2.1 理论计算题

输入一张大小为 3 × 32 × 32（通道数 × 高 × 宽）的彩色图像。通过一
个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 3 × 5 × 5。设定填
充（Padding）为 2，步幅（Stride）为 2。
1. 请计算该卷积层输出的特征图（Feature Map）的尺寸（通道数 × 高
× 宽）。
2. 计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行
多少次点乘（乘法）操作？

##### 卷积计算解答
##### 已知参数
输入：$C_{in}=3,\ H_{in}=32,\ W_{in}=32$
卷积核：$C_{out}=16,\ k_h=5,\ k_w=5$
$Padding\ P=2,\ Stride\ S=2$

输出尺寸公式：
$$
H_{out}=\left\lfloor \frac{H_{in}+2P-k_h}{S}+1 \right\rfloor,\quad
W_{out}=\left\lfloor \frac{W_{in}+2P-k_w}{S}+1 \right\rfloor
$$

##### 1. 输出特征图尺寸
\[
H_{out}=\frac{32+2\times2-5}{2}+1
=\frac{32+4-5}{2}+1
=\frac{31}{2}+1=15.5+1 \xrightarrow{\lfloor\cdot\rfloor}=16
\]
\[
W_{out}=16
\]
输出通道数 = 卷积核个数 = $16$

**输出尺寸：$\boldsymbol{16\times16\times16}$**

##### 2. 单个输出像素乘法次数
单个卷积核：$C_{in}\times k_h\times k_w$，一次卷积点乘次数：
\[
3\times5\times5 = 75
\]
**答案：75次乘法**

### 2.2 编程题

不使用深度学习框架的底层 Pooling API（如 ‘torch.nn.MaxPool2d‘），
仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支
持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向
传播函数。

In [1]:
import numpy as np

def max_pool2d_forward(X, pool_size, stride=1, padding=0):
    """
    手动实现 2D 最大池化前向传播
    参数：
        X: 输入特征图，形状为 (N, C, H, W)
            N: batch大小
            C: 通道数
            H: 高度
            W: 宽度
        pool_size: 池化核大小 (kh, kw)
        stride: 步幅 (sh, sw)
        padding: 填充大小 (ph, pw)
    返回：
        out: 池化后的特征图
        cache: 保存输入、最大值位置（用于反向传播）
    """
    # 解析参数
    N, C, H, W = X.shape
    kh, kw = pool_size
    sh, sw = stride if isinstance(stride, tuple) else (stride, stride)
    ph, pw = padding if isinstance(padding, tuple) else (padding, padding)

    # 填充：上下左右各 pad ph, pw 个 0
    X_padded = np.pad(X, ((0, 0), (0, 0), (ph, ph), (pw, pw)), mode='constant')

    # 计算输出尺寸
    H_out = (H + 2 * ph - kh) // sh + 1
    W_out = (W + 2 * pw - kw) // sw + 1

    # 初始化输出
    out = np.zeros((N, C, H_out, W_out))

    # 逐元素计算最大池化
    for n in range(N):
        for c in range(C):
            for h in range(H_out):
                for w in range(W_out):
                    # 定位当前窗口
                    h_start = h * sh
                    h_end = h_start + kh
                    w_start = w * sw
                    w_end = w_start + kw

                    # 取出窗口并取最大值
                    window = X_padded[n, c, h_start:h_end, w_start:w_end]
                    out[n, c, h, w] = np.max(window)

    return out

if __name__ == "__main__":
    # 构造输入：1个样本，3个通道，8×8
    X = np.random.randn(1, 3, 8, 8)

    # 最大池化：核2×2，步幅2，填充0
    out = max_pool2d_forward(X, pool_size=(2, 2), stride=2, padding=0)

    print("输入形状:", X.shape)
    print("输出形状:", out.shape)

输入形状: (1, 3, 8, 8)
输出形状: (1, 3, 4, 4)


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

在 VGG 网络中，作者频繁使用多个 3 × 3 卷积核级联来代替较大的卷
积核（如 5 × 5 或 7 × 7）。假设输入和输出的特征图通道数均为 C。
1. 计算一个 5 × 5 卷积层（不带偏置）的参数量。
2. 计算两个串联的 3 × 3 卷积层（不带偏置，两层通道数都为 C）的总
参数量。

#### VGG 卷积参数量计算
> 条件：输入输出通道均为 $C$，无偏置参数，参数量 = 卷积核总权重数量

#### 1. 单个 $5\times5$ 卷积参数量
卷积核尺寸：$C_{in}=C,\ k=5\times5,\ C_{out}=C$
$$
Param_{5\times5}=C\times5\times5\times C=\boldsymbol{25C^2}
$$

#### 2. 两层串联 $3\times3$ 卷积总参数量
两层通道保持均为 $C$：
- 第一层：$C\times3\times3\times C=9C^2$
- 第二层：$C\times3\times3\times C=9C^2$

$$
Param_{two\times3\times3}=9C^2+9C^2=\boldsymbol{18C^2}
$$

#### 结论
同等感受野下，两个串联3×3参数量 $18C^2$ ＜ 单个5×5参数量 $25C^2$，参数更少、非线性更强。

### 3.2 编程题

NiN 网络的核心创新是引入了“1x1 卷积”组成的 NiN 块来代替传统
的全连接层，以减少参数量。请使用 PyTorch（‘torch.nn.Sequential‘）定义
一个标准的 NiN 块（NiN Block）。  
要求：NiN 块接收输入通道数 ‘in_channels‘ 和输出通道数 ‘out_channels‘，
它由一个普通的卷积层（指定窗口大小 ‘kernel_size‘，步幅 ‘stride‘，填充
‘padding‘）以及两个随后的 1 × 1 卷积层级联组成。每层卷积后都需要紧跟
一个 ReLU 激活层。

In [2]:
import torch
import torch.nn as nn

# 定义 NiN 块（NiN Block）
class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 第一层：普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            
            # 第二层：1×1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            
            # 第三层：1×1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)
    
# 构造输入：batch=1, 通道=3, 高=32, 宽=32
x = torch.randn(1, 3, 32, 32)

# 创建 NiN 块
nin_block = NiNBlock(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1
)

# 前向传播
out = nin_block(x)
print("输出形状:", out.shape)

输出形状: torch.Size([1, 16, 32, 32])


## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置
的特征值在 4 个样本上的输出分别为：x1 = 2, x2 = 4, x3 = 6, x4 = 8。假设
当前批量归一化层学到的缩放参数 γ = 2，平移参数 β = 1，常数 ϵ = 0。  
请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值
y1, y2, y3, y4。

#### BN批量归一化计算
#### 已知
$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8,\quad \gamma=2,\ \beta=1,\ \epsilon=0$
BN公式：
$$
\hat x_i = \frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}},\quad y_i=\gamma \hat x_i+\beta
$$

#### 1. 求均值$\mu$
$$
\mu=\frac{2+4+6+8}{4}=5
$$

#### 2. 求方差$\sigma^2$
$$
\begin{align}
\sigma^2 &= \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}\\
&=\frac{9+1+1+9}{4}=5
\end{align}
$$
$\sqrt{\sigma^2+\epsilon}=\sqrt5$

#### 3. 逐个计算输出
$$
\begin{aligned}
y_1 &= 2\cdot\frac{2-5}{\sqrt5}+1 = -\frac{6}{\sqrt5}+1 \\
y_2 &= 2\cdot\frac{4-5}{\sqrt5}+1 = -\frac{2}{\sqrt5}+1 \\
y_3 &= 2\cdot\frac{6-5}{\sqrt5}+1 = \frac{2}{\sqrt5}+1 \\
y_4 &= 2\cdot\frac{8-5}{\sqrt5}+1 = \frac{6}{\sqrt5}+1 \\
\end{aligned}
$$

**结果：**
$y_1=1-\dfrac{6}{\sqrt5},\ y_2=1-\dfrac{2}{\sqrt5},\ y_3=1+\dfrac{2}{\sqrt5},\ y_4=1+\dfrac{6}{\sqrt5}$

### 4.2 编程题

残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的
梯度消失问题。请用 PyTorch 自定义一个残差块类 ‘Residual‘。  
要求：该块包含两个具有相同输出通道数的 3 × 3 卷积层，每个卷积层
后跟一个批量归一化层。如果 ‘use_1x1conv=True‘，则需要对输入应用一
个 1 × 1 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输
出进行按元素相加（f(x) + x）。

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()
        
        # 主路径：两个 3x3 卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 捷径（shortcut）：如果需要，用 1x1 卷积调整通道/形状
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                                   stride=stride)
        
        # 激活函数
        self.relu = nn.ReLU()

    def forward(self, x):
        # 主分支输出
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 捷径分支
        if self.conv3 is not None:
            x = self.conv3(x)
        
        # 残差相加 f(x) + x
        y += x
        
        # 最后激活
        y = self.relu(y)
        return y

# 输入：batch=1, channels=3, height=32, width=32
x = torch.randn(1, 3, 32, 32)

# 创建残差块（需要 1x1 卷积来匹配通道）
block = Residual(in_channels=3, out_channels=16, use_1x1conv=True)
out = block(x)

print("输出形状:", out.shape)

输出形状: torch.Size([1, 16, 32, 32])


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如
ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。请
回答以下关于微调理论的问题：
1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较
小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出
层”设置较大的学习率？
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么
样的微调策略以防止过拟合？

#### 微调(Fine-tuning)简答题答案
#### 1. 底层小学习率、顶层输出层大学习率的原因
1. **底层卷积层作用：通用基础特征**
预训练模型底层（浅层）已经在ImageNet海量数据上学到边缘、纹理、轮廓等通用视觉特征，这些特征具备通用性，适配大部分图像任务，不需要大幅改动参数。
- 冻结/小学习率：避免破坏已经训练成熟的通用特征；
- 若设置大学习率，原有优质参数被大幅度更新，丢失预训练权重的先验知识。

2. **顶层分类层作用：任务专属特征**
顶层输出层是针对原数据集分类任务训练的，**全新数据集类别、标签完全不同，输出层参数随机初始化**，没有有效先验信息，需要快速收敛适配新任务。
- 设置大学习率：加快新参数学习，快速拟合目标数据集分布。

总结：底层保留预训练通用特征（慢更新/冻结），顶层从零学习新任务分类规则（快速更新）。

#### 2. 目标数据集很小、和源数据集高度相似的微调策略（防过拟合）
1. **策略一：冻结全部特征提取主干，只训练最后新增的输出层**
主干网络参数完全固定不参与更新，仅训练随机初始化的全连接/分类头。利用预训练成熟特征做特征提取，小数据集只需学习简单映射关系，最大限度避免过拟合。

2. **策略二：全网络使用极小学习率微调**
若少量微调主干，整体设置极低学习率，权重仅在预训练参数附近小幅震荡微调，不会大幅度偏离原有权重，抑制小样本带来的过拟合。

3. **配套正则化手段**
加入Dropout、L2正则、数据增强（随机裁剪、翻转），扩充有效样本，进一步降低过拟合风险。

### 5.2 编程题

图像增广能有效增强模型的泛化能力。请利用 ‘torchvision.transforms‘
模块创建一个组合图像增广管道（Pipeline）。
1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后
的图像缩放到 224 × 224 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度
（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。

In [5]:
import torchvision.transforms as transforms

# 构建图像增广Pipeline
train_transform = transforms.Compose([
    # 1.随机裁剪，面积0.08~1.0，缩放至224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2.50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3.亮度、对比度、饱和度随机扰动0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4.转为张量
    transforms.ToTensor()
])

## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重
合程度。已知图像中两个边界框（以 ‘[左上角 x, 左上角 y, 右下角 x, 右下
角 y]‘ 格式表示）：
1. 真实框（Ground Truth）A = [10, 10, 50, 50]
2. 预测框（Prediction Box）B = [30, 30, 70, 70]
请计算边界框 A 和边界框 B 之间的 IoU 准确值。

#### IoU交并比计算
#### 已知
$A=[x_{A1},y_{A1},x_{A2},y_{A2}]=[10,10,50,50]$
$B=[x_{B1},y_{B1},x_{B2},y_{B2}]=[30,30,70,70]$

#### 1. 相交区域坐标
交集左上角：
$$x_{in1}=\max(10,30)=30,\quad y_{in1}=\max(10,30)=30$$
交集右下角：
$$x_{in2}=\min(50,70)=50,\quad y_{in2}=\min(50,70)=50$$

相交宽高：
$$w_{in}=50-30=20,\quad h_{in}=50-30=20$$
相交面积：
$$S_{in}=20\times20=400$$

#### 2. 分别计算A、B面积
$$S_A=(50-10)\times(50-10)=40\times40=1600$$
$$S_B=(70-30)\times(70-30)=40\times40=1600$$

#### 3. 并集面积
$$S_{union}=S_A+S_B-S_{in}=1600+1600-400=2800$$

#### 4. IoU
$$IoU=\frac{S_{in}}{S_{union}}=\frac{400}{2800}=\frac{1}{7}$$

**答案：$\boldsymbol{IoU=\dfrac{1}{7}\approx0.1429}$**

### 6.2 编程题

在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模
型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（Onehot），若设置平滑因子 ϵ = 0.1，则对于 K 分类问题，真实标签对应的目标
概率从 1 变为 1 − ϵ，其余错误类别的概率从 0 变为 K
ϵ
−1。  
请实现一个计算标签平滑后交叉熵损失的函数。

In [6]:
import torch
import torch.nn as nn

def label_smoothing_cross_entropy(preds, targets, epsilon=0.1, num_classes=10):
    """
    标签平滑后的交叉熵损失
    Args:
        preds: 模型输出 logits (batch_size, num_classes)
        targets: 真实标签 (batch_size,)
        epsilon: 平滑因子
        num_classes: 分类类别数 K
    """
    # 计算平滑后的标签分布
    one_hot = torch.zeros_like(preds).scatter(1, targets.unsqueeze(1), 1)
    smooth_label = one_hot * (1 - epsilon) + epsilon / num_classes

    # 计算 log softmax
    log_preds = torch.log_softmax(preds, dim=1)

    # 交叉熵损失
    loss = -torch.sum(smooth_label * log_preds, dim=1).mean()
    return loss

# 模拟输入
preds = torch.randn(8, 10)  # batch=8, 10分类
targets = torch.randint(0, 10, (8,))

# 计算损失
loss = label_smoothing_cross_entropy(preds, targets, epsilon=0.1, num_classes=10)
print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 1.6267231702804565
